# Phase D / Experiment 2 — Embedding Model (RQ2)

Implements **MASTER_PLAN.md §10 D.2** (embedding-model sweep at the Exp1
winner config) under the conventions in §13.5 (determinism) and §15.

**Frozen baseline (from the locked Exp1 winner `aadb0cb9`):**
- `chunk_size = 512`
- `overlap_pct = 20`
- `top_k = 3`
- `llm = llama3.1:8b-instruct-q4_K_M`, `temperature = 0`, `seed = 42`

**Varied axis — embedding model:**

| # | model | dim | strategy |
|---|-------|----:|----------|
| 1 | `sentence-transformers/all-MiniLM-L6-v2`    |  384 | build fresh (rebuilds `indices/aadb0cb9/` bit-identically — deterministic — but produces a clean Exp2-internal indexing measurement under the same OfflineEmissionsTracker as the other two configs) |
| 2 | `sentence-transformers/all-mpnet-base-v2`   |  768 | build fresh |
| 3 | `BAAI/bge-large-en-v1.5`                     | 1024 | build fresh, with RAM pre-flight against the Llama 3.1 8B Q4 resident footprint (§14 risk register) |

**Per config:** 1 indexing row + 3 reps × 13 q = 40 rows.
**Exp2 total:** 3 indexing rows + 117 query rows.

**MiniLM reuse policy (override).** An earlier draft of this notebook
reused `indices/aadb0cb9/` for the MiniLM config and copied indexing
measurements from the matching `phase_d_exp1` row. The user opted to
**rebuild fresh instead** (this run) so all three Exp2 indexing rows
are measured under the same `OfflineEmissionsTracker` and
`PeakRAMSampler` conditions — methodologically cleaner for the
embedding-model comparison. The rebuild is deterministic (chunking +
CPU sentence-transformers + seed 42 → bit-identical artifact), so the
on-disk index is unchanged in content; only the indexing-time wall /
energy / RAM measurements are fresh. Cell 4 reports the auto-detected
reuse-eligibility for diagnostic purposes but unconditionally
overrides to `reuse_decisions = [False, False, False]`. The reuse
code path in cell 5 is preserved (not exercised this run) for future
runs that may want it.

**Run-id scheme (carried forward from Exp1):**
- indexing run_id = `{ts_compact}_{config_hash}`
- query   run_id = `{indexing_run_id}_r{r}` for r ∈ {1, 2, 3}

**CodeCarbon (§13.4).** `OfflineEmissionsTracker` for every block.
`project_name=phase_d_exp2_indexing_<hash>` and
`project_name=phase_d_exp2_query_<hash>_r{r}`. The reused MiniLM config
does NOT open an indexing tracker (no real work to measure); its query
reps DO each open a tracker as usual.

**RAM pre-flight (§14 risk register).** Before each fresh indexing
build, cell 5's `ram_preflight()` projects post-load resident memory and
halts if the projection exceeds `RAM_HALT_THRESHOLD_MB` (default 12 GB).
With reuse turned off this run, the pre-flight runs on all 3 configs.
MiniLM is the cheapest (~120 MB estimate); MPNet is medium (~460 MB);
BGE-large is the binding case (~1400 MB).

**Best-Exp2-config rule (locked, mirrors Exp1).** Lexicographic Pareto:

1. primary: lowest mean refusal rate over the 3 reps (refusal := answer
   contains the literal phrase `"I don't have enough information"`).
2. tie-break #1: lowest mean `total_latency_ms` over (3 reps × 13 q).
3. tie-break #2: lowest `indexing_time_sec`.

The Exp2 winner's `embedding_model` is then frozen as the Exp3 baseline
in `notebooks/05_exp3_topk_depth.ipynb`.


In [1]:
"""Cell 2 — Loop driver, frozen baseline, RAM halt threshold.

A single Run All sweeps every model in EMBEDDING_MODELS_TO_RUN against the
frozen Exp1 winner baseline (chunk_size=512, overlap_pct=20%, top_k=3).

NOTE — partial-completion state (2026-05-10).
MiniLM (config_hash=aadb0cb9) was already completed in the prior Run All
on 2026-05-10 at 09:17:35Z; its rows are already persisted in
  - results/indexing_log.csv  (1 row, run_id=20260510T091735Z_aadb0cb9,
                               notes="phase_d_exp2")
  - results/query_log.csv     (39 rows under aadb0cb9_r{1,2,3})
That run was halted by the RAM pre-flight on MPNet (used_now=14050 MB
exceeded the 12288 MB threshold); the user has since restarted the Mac to
free RAM. Do NOT re-include MiniLM in this tuple — that would create
duplicate config_hash measurements under a fresh Exp2 run_id and inflate
the 117-row Exp2 query budget. The remaining sweep below covers MPNet and
BGE-large only; cell 7 reads phase_d_exp2 rows from CSV at the end so the
final Exp2 Pareto ranking covers all 3 configs (MiniLM from the partial
run + MPNet/BGE from this run).
"""

# §10 D.2 axis: embedding model. MiniLM intentionally excluded — see
# the docstring above.
EMBEDDING_MODELS_TO_RUN: tuple[str, ...] = (
    "sentence-transformers/all-mpnet-base-v2",    # 768-d
    "BAAI/bge-large-en-v1.5",                     # 1024-d (RAM-watched)
)

# Frozen from Exp1 winner aadb0cb9.
FROZEN_CHUNK_SIZE:  int = 512
FROZEN_OVERLAP_PCT: int = 20
EXP1_WINNER_HASH: str = "aadb0cb9"

# §14 RAM halt threshold. Pre-flight projects (current_used + model_size +
# embedding_spike_buffer) and halts if it exceeds this number, on any
# fresh indexing build. 12 GB leaves ~4 GB of headroom on a 16 GB M4.
RAM_HALT_THRESHOLD_MB: int = 12 * 1024  # 12 GB

# Conservative resident-footprint estimates for each embedding model.
# Used by ram_preflight (cell 5). Includes model weights + a runtime
# overhead margin observed empirically with sentence-transformers on
# Apple Silicon CPU. The runtime pre-flight is the binding check; these
# estimates only drive the projection.
MODEL_SIZE_ESTIMATES_MB: dict[str, float] = {
    "sentence-transformers/all-MiniLM-L6-v2":  120.0,
    "sentence-transformers/all-mpnet-base-v2": 460.0,
    "BAAI/bge-large-en-v1.5":                 1400.0,
}


In [2]:
"""Cell 3 — Imports, paths, determinism guards, eval questions.

One-time setup. Same scaffolding as 03_exp1_chunk_overlap.ipynb cell 3,
plus a list-level validation of EMBEDDING_MODELS_TO_RUN.
"""
from __future__ import annotations

import random
import sys
import time
import tracemalloc
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

_HERE = Path.cwd()
ROOT = _HERE if (_HERE / "MASTER_PLAN.md").exists() else _HERE.parent
assert (ROOT / "MASTER_PLAN.md").exists(), f"Could not find project root from {_HERE}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.chunking import chunk_corpus
from src.embedding import embed_chunks
from src.eval_questions import load_eval_questions
from src.indexing import build_index, load_index, save_index
from src.metrics import (
    EXPECTED_INDEXING_COLS,
    EXPECTED_QUERY_COLS,
    NOTES_TAG_PROP_ENERGY,
    PeakRAMSampler,
    allocate_block_energy_proportionally,
    assert_indexing_schema,
    assert_query_sanity,
    assert_query_schema,
    count_retrieved_tokens,
    count_text_tokens,
    make_run_id,
    set_global_seeds,
    timing_context,
)
from src.pipeline import RAGConfig, run_rag

# §9 C.5 — determinism guards (notebook-side, in addition to Ollama defaults).
random.seed(42)
np.random.seed(42)
set_global_seeds(42)

# Validate the driver tuple (cell 2).
assert isinstance(EMBEDDING_MODELS_TO_RUN, tuple), (
    "EMBEDDING_MODELS_TO_RUN must be a tuple of strings (cell 2)"
)
assert len(EMBEDDING_MODELS_TO_RUN) >= 1, (
    "EMBEDDING_MODELS_TO_RUN is empty — nothing to run"
)
assert len(set(EMBEDDING_MODELS_TO_RUN)) == len(EMBEDDING_MODELS_TO_RUN), (
    f"duplicate values in EMBEDDING_MODELS_TO_RUN={EMBEDDING_MODELS_TO_RUN}"
)
for _m in EMBEDDING_MODELS_TO_RUN:
    assert _m in MODEL_SIZE_ESTIMATES_MB, (
        f"missing MODEL_SIZE_ESTIMATES_MB entry for {_m!r}"
    )

# Project paths.
RESULTS_DIR  = ROOT / "results"
CARBON_DIR   = RESULTS_DIR / "carbon"
INDICES_DIR  = ROOT / "indices"
CORPUS_TXT   = ROOT / "corpus_text"
EVAL_QS_PATH = ROOT / "dataset" / "eval_questions.md"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CARBON_DIR.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)

# Load the 13 eval questions — same parser used by Phase B/C/D-Exp1.
eval_questions = load_eval_questions(EVAL_QS_PATH)
type_counts: dict[str, int] = {}
for q in eval_questions:
    type_counts[q.question_type] = type_counts.get(q.question_type, 0) + 1

vm0 = psutil.virtual_memory()
print(f"[D.2] Project root:          {ROOT}")
print(f"[D.2] Frozen baseline:       chunk_size={FROZEN_CHUNK_SIZE}, "
      f"overlap_pct={FROZEN_OVERLAP_PCT}%, top_k=3 (Exp1 winner aadb0cb9)")
print(f"[D.2] Embedding sweep:       {len(EMBEDDING_MODELS_TO_RUN)} models")
print(f"[D.2] RAM halt threshold:    {RAM_HALT_THRESHOLD_MB} MB")
print(f"[D.2] Eval questions:        {len(eval_questions)} ; by type: {type_counts}")
print(f"[D.2] Memory at notebook startup: total={vm0.total/1024**2:.0f} MB, "
      f"available={vm0.available/1024**2:.0f} MB, "
      f"used={(vm0.total-vm0.available)/1024**2:.0f} MB ({vm0.percent:.1f}%)")


[D.2] Project root:          <project_root>
[D.2] Frozen baseline:       chunk_size=512, overlap_pct=20%, top_k=3 (Exp1 winner aadb0cb9)
[D.2] Embedding sweep:       2 models
[D.2] RAM halt threshold:    12288 MB
[D.2] Eval questions:        13 ; by type: {'factoid': 6, 'synthesis': 3, 'numerical': 3, 'multi-doc': 1}
[D.2] Memory at notebook startup: total=16384 MB, available=7008 MB, used=9376 MB (57.2%)


In [3]:
"""Cell 4 — Pre-flight: build the 3 RAGConfigs, compute config_hashes,
detect which configs can REUSE an existing on-disk artifact.

Reuse criterion (per the user's "reuse, do NOT re-index" instruction for
MiniLM): a config is reusable iff
    (a) results/indexing_log.csv has a row with this config_hash AND
        notes == "phase_d_exp1", AND
    (b) indices/<config_hash>/{faiss.index, chunks.parquet} exist on disk.

Either condition failing → fall back to a fresh build. This keeps reuse
opt-in and self-validating: a manual reset of CSVs or indices/ would
quietly switch to fresh builds with no methodological surprise.
"""

configs_to_run: list[RAGConfig] = [
    RAGConfig(
        chunk_size=FROZEN_CHUNK_SIZE,
        overlap_pct=FROZEN_OVERLAP_PCT,
        embedding_model=m,
    )
    for m in EMBEDDING_MODELS_TO_RUN
]

# Hash collisions across the 3 configs would indicate a logic error.
_hashes = [c.hash for c in configs_to_run]
assert len(set(_hashes)) == len(_hashes), (
    f"BUG: duplicate config_hash among Exp2 configs: "
    f"{list(zip(_hashes, [c.embedding_model for c in configs_to_run]))}"
)

# Read existing indexing_log.csv once to detect Exp1 rows we can reuse.
_idx_log_path = RESULTS_DIR / "indexing_log.csv"
if _idx_log_path.exists():
    _existing_idx = pd.read_csv(_idx_log_path)
else:
    _existing_idx = pd.DataFrame(columns=EXPECTED_INDEXING_COLS)

# Auto-detect which configs would be eligible for reuse (artifact on disk
# AND matching phase_d_exp1 indexing row). Reported as a diagnostic, but
# this run unconditionally OVERRIDES eligibility to all-False so every
# Exp2 indexing row is freshly measured under the same OfflineEmissionsTracker
# and PeakRAMSampler conditions (per user instruction this session;
# "rebuild MiniLM" — see cell 1's MiniLM reuse-policy paragraph).
_reuse_eligible: list[bool] = []
print(f"[D.2] Pre-flight: {len(configs_to_run)} configs to sweep")
for cfg in configs_to_run:
    artifact_dir = INDICES_DIR / cfg.hash
    artifact_ok = (
        (artifact_dir / "faiss.index").exists()
        and (artifact_dir / "chunks.parquet").exists()
    )
    exp1_match = _existing_idx[
        (_existing_idx["config_hash"] == cfg.hash)
        & (_existing_idx["notes"] == "phase_d_exp1")
    ]
    has_exp1_row = len(exp1_match) >= 1
    eligible = artifact_ok and has_exp1_row
    _reuse_eligible.append(eligible)
    tag = "BUILD fresh"
    diag = " (reuse-eligible — overridden)" if eligible else ""
    est_mb = MODEL_SIZE_ESTIMATES_MB.get(cfg.embedding_model, float('nan'))
    print(f"  hash={cfg.hash}  embedding={cfg.embedding_model:<48} "
          f"est_model={est_mb:>6.0f} MB   {tag}{diag}")

# Reuse override (user instruction this session). All three configs build
# fresh — see cell 1 "MiniLM reuse policy (override)" for the rationale.
reuse_decisions: list[bool] = [False] * len(configs_to_run)

print()
print(f"[D.2] reuse_eligible (auto-detected) = {_reuse_eligible}")
print(f"[D.2] reuse_decisions (after override) = {reuse_decisions}")


[D.2] Pre-flight: 2 configs to sweep
  hash=d7129fb0  embedding=sentence-transformers/all-mpnet-base-v2          est_model=   460 MB   BUILD fresh
  hash=4d57f78b  embedding=BAAI/bge-large-en-v1.5                           est_model=  1400 MB   BUILD fresh

[D.2] reuse_eligible (auto-detected) = [False, False]
[D.2] reuse_decisions (after override) = [False, False]


In [4]:
"""Cell 5 — Helpers.

ram_preflight(): given an estimated model size, project total used memory
after a load + an embedding spike buffer; halt if the projection exceeds
RAM_HALT_THRESHOLD_MB. Called immediately before each fresh indexing
build.

run_one_exp2_config(): per-config end-to-end runner. Reuses Exp1's
run_one_config flow but with two adjustments:
  - if reuse_artifact=True: skip the indexing build and emit the indexing
    row by copying measurements from the matching phase_d_exp1 row;
  - else: run the RAM pre-flight, then build the index inside the usual
    OfflineEmissionsTracker / PeakRAMSampler / timing_context wrappers.

All 3 query reps (per §10 D.4) run identically regardless of whether the
indexing was reused or built fresh — each rep opens its own
OfflineEmissionsTracker (project_name=phase_d_exp2_query_<hash>_r{r}).
"""
from codecarbon import OfflineEmissionsTracker

REFUSAL_PHRASE = "I don't have enough information"
EMBEDDING_SPIKE_BUFFER_MB = 500.0


def ram_preflight(
    model_size_mb_estimate: float,
    *,
    halt_threshold_mb: int = RAM_HALT_THRESHOLD_MB,
) -> tuple[bool, str]:
    """Project resident-memory usage after model load + embedding spike.

    Returns (ok, message). On ok=False, the calling cell must halt and
    let the user decide whether to drop the model.
    """
    vm = psutil.virtual_memory()
    used_mb_now = (vm.total - vm.available) / 1024**2
    proj_used_mb = used_mb_now + model_size_mb_estimate + EMBEDDING_SPIKE_BUFFER_MB
    proj_avail_mb = (vm.total / 1024**2) - proj_used_mb
    msg = (
        f"used_now={used_mb_now:.0f} MB, model_est={model_size_mb_estimate:.0f}, "
        f"spike_buffer={EMBEDDING_SPIKE_BUFFER_MB:.0f} -> "
        f"proj_used={proj_used_mb:.0f} MB (threshold {halt_threshold_mb} MB), "
        f"proj_available={proj_avail_mb:.0f} MB"
    )
    return proj_used_mb < halt_threshold_mb, msg


def run_one_exp2_config(cfg: RAGConfig, *, reuse_artifact: bool) -> dict:
    # 1) UTC anchor — shared by indexing run_id and all 3 query run_ids.
    config_ts = datetime.now(timezone.utc)
    idx_ts_iso, idx_ts_compact, idx_run_id = make_run_id(
        cfg.hash, ts=config_ts,
    )

    print()
    print("=" * 78)
    print(f"[D.2] Config: embedding={cfg.embedding_model}")
    print(f"      chunk_size={cfg.chunk_size}, overlap_pct={cfg.overlap_pct}%, "
          f"hash={cfg.hash}, reuse_artifact={reuse_artifact}")
    print(f"      idx_run_id = {idx_run_id}")
    print("=" * 78)

    INDEX_DIR = INDICES_DIR / cfg.hash

    if reuse_artifact:
        # No rebuild, no indexing tracker. Load the artifact and synthesise
        # an indexing row by copying measurements from the phase_d_exp1 row.
        faiss_index, chunks = load_index(INDEX_DIR)
        idx_csv = RESULTS_DIR / "indexing_log.csv"
        existing = pd.read_csv(idx_csv)
        e1 = existing[
            (existing["config_hash"] == cfg.hash)
            & (existing["notes"] == "phase_d_exp1")
        ]
        assert len(e1) == 1, (
            f"reuse_artifact=True for hash={cfg.hash}, but found "
            f"{len(e1)} phase_d_exp1 indexing rows (expected exactly 1)"
        )
        e1 = e1.iloc[0]

        indexing_row = {
            "run_id":               idx_run_id,
            "timestamp":            idx_ts_iso,
            "config_hash":          cfg.hash,
            "chunk_size":           int(cfg.chunk_size),
            "chunk_overlap":        int(cfg.overlap_pct),
            "embedding_model":      cfg.embedding_model,
            "n_documents":          int(e1["n_documents"]),
            "n_chunks_total":       int(e1["n_chunks_total"]),
            "indexing_time_sec":    float(e1["indexing_time_sec"]),
            "peak_ram_mb":          float(e1["peak_ram_mb"]),
            "index_size_mb":        float(e1["index_size_mb"]),
            "embedding_time_sec":   float(e1["embedding_time_sec"]),
            "faiss_build_time_sec": float(e1["faiss_build_time_sec"]),
            "energy_wh":            float(e1["energy_wh"]),
            "co2_g":                float(e1["co2_g"]),
            "notes":                "phase_d_exp2 (reused phase_d_exp1 "
                                    f"{cfg.hash} indexing measurements)",
        }
        idx_energy_wh = float(e1["energy_wh"])
        idx_co2_g     = float(e1["co2_g"])
        INDEXING_TIME_SEC = float(e1["indexing_time_sec"])
        peak_ram_mb_for_idx = float(e1["peak_ram_mb"])
        INDEX_SIZE_MB = float(e1["index_size_mb"])
        n_chunks_for_summary = int(e1["n_chunks_total"])

        print(f"[D.2] REUSED artifact at {INDEX_DIR} (n_chunks={len(chunks)}, "
              f"file_size={INDEX_SIZE_MB:.2f} MB).")
        print(f"      Copied phase_d_exp1 measurements: "
              f"indexing_time_sec={INDEXING_TIME_SEC:.2f}, "
              f"peak_ram_mb={peak_ram_mb_for_idx:.1f}, "
              f"energy_wh={idx_energy_wh:.4f}, co2_g={idx_co2_g:.4f}.")

    else:
        # Fresh build inside the usual measurement plumbing. RAM pre-flight
        # first (§14 risk register).
        est_mb = MODEL_SIZE_ESTIMATES_MB[cfg.embedding_model]
        ok, ram_msg = ram_preflight(est_mb)
        print(f"[D.2] RAM pre-flight: {ram_msg}")
        if not ok:
            raise RuntimeError(
                f"RAM pre-flight HALT for {cfg.embedding_model}: {ram_msg}"
            )

        idx_tracker = OfflineEmissionsTracker(
            project_name=f"phase_d_exp2_indexing_{cfg.hash}",
            tracking_mode="machine",
            country_iso_code="DEU",
            output_dir=str(CARBON_DIR),
            output_file="emissions.csv",
            measure_power_secs=1.0,
            log_level="error",
            save_to_file=True,
        )
        tracemalloc.start()
        idx_tracker.start()
        try:
            with timing_context() as IDX_T, PeakRAMSampler(interval_sec=0.1) as RAM:
                chunks = chunk_corpus(
                    text_dir=CORPUS_TXT,
                    chunk_size=cfg.chunk_size,
                    overlap_pct=cfg.overlap_pct,
                )
                embeddings = embed_chunks(
                    chunks,
                    model_name=cfg.embedding_model,
                    show_progress=False,
                )
                faiss_index = build_index(embeddings)
                save_index(faiss_index, chunks, INDEX_DIR)
        finally:
            idx_co2_kg = idx_tracker.stop()
            _, tm_peak_b = tracemalloc.get_traced_memory()
            tracemalloc.stop()

        idx_energy_wh = idx_tracker.final_emissions_data.energy_consumed * 1000.0
        idx_co2_g     = idx_co2_kg * 1000.0
        INDEX_SIZE_MB = (INDEX_DIR / "faiss.index").stat().st_size / 1_000_000.0
        INDEXING_TIME_SEC = (
            IDX_T.get("chunking", 0.0)
            + IDX_T.get("embedding", 0.0)
            + IDX_T.get("faiss_build", 0.0)
            + IDX_T.get("index_save", 0.0)
        )
        peak_ram_mb_for_idx = float(RAM.peak_rss_mb)
        n_chunks_for_summary = len(chunks)

        n_documents = len({c.source_doc for c in chunks})
        indexing_row = {
            "run_id":               idx_run_id,
            "timestamp":            idx_ts_iso,
            "config_hash":          cfg.hash,
            "chunk_size":           int(cfg.chunk_size),
            "chunk_overlap":        int(cfg.overlap_pct),
            "embedding_model":      cfg.embedding_model,
            "n_documents":          int(n_documents),
            "n_chunks_total":       int(len(chunks)),
            "indexing_time_sec":    float(INDEXING_TIME_SEC),
            "peak_ram_mb":          float(peak_ram_mb_for_idx),
            "index_size_mb":        float(INDEX_SIZE_MB),
            "embedding_time_sec":   float(IDX_T.get("embedding", 0.0)),
            "faiss_build_time_sec": float(IDX_T.get("faiss_build", 0.0)),
            "energy_wh":            float(idx_energy_wh),
            "co2_g":                float(idx_co2_g),
            "notes":                "phase_d_exp2",
        }

        print(f"[D.2] Indexed: n_chunks={len(chunks)}, "
              f"indexing_time={INDEXING_TIME_SEC:.2f}s, "
              f"peak_ram={peak_ram_mb_for_idx:.1f} MB, "
              f"index_size={INDEX_SIZE_MB:.2f} MB, "
              f"energy={idx_energy_wh:.4f} Wh, "
              f"tracemalloc_peak={tm_peak_b/1_000_000:.2f} MB (Q4: not in CSV)")

    # Append the indexing row.
    idx_csv = RESULTS_DIR / "indexing_log.csv"
    write_header = not idx_csv.exists()
    pd.DataFrame([indexing_row], columns=EXPECTED_INDEXING_COLS).to_csv(
        idx_csv, mode="a", index=False, header=write_header,
    )

    # 2) Three query reps — identical structure to Exp1 cell 5.
    rep_summaries: list[dict] = []
    for r in (1, 2, 3):
        rep_ts_iso, _, qry_run_id = make_run_id(
            cfg.hash, ts=config_ts, repetition=r,
        )
        assert qry_run_id.startswith(idx_run_id), (
            f"BUG: qry_run_id={qry_run_id!r} not prefixed by "
            f"idx_run_id={idx_run_id!r}"
        )

        qry_tracker = OfflineEmissionsTracker(
            project_name=f"phase_d_exp2_query_{cfg.hash}_r{r}",
            tracking_mode="machine",
            country_iso_code="DEU",
            output_dir=str(CARBON_DIR),
            output_file="emissions.csv",
            measure_power_secs=1.0,
            log_level="error",
            save_to_file=True,
        )

        rows: list[dict] = []
        rep_t0 = time.perf_counter()
        qry_tracker.start()
        try:
            for q in eval_questions:
                result = run_rag(q.question, cfg, faiss_index, chunks)
                t = result["timings"]
                retrieved = result["retrieved_chunks"]

                embed_ms    = t["embed"]    * 1000.0
                retrieve_ms = t["retrieve"] * 1000.0
                generate_ms = t["generate"] * 1000.0
                total_ms    = embed_ms + retrieve_ms + generate_ms

                rows.append({
                    "run_id":                qry_run_id,
                    "timestamp":             rep_ts_iso,
                    "config_hash":           cfg.hash,
                    "question_id":           q.question_id,
                    "top_k":                 int(cfg.top_k),
                    "query_embed_time_ms":   float(embed_ms),
                    "retrieval_time_ms":     float(retrieve_ms),
                    "generation_time_ms":    float(generate_ms),
                    "total_latency_ms":      float(total_ms),
                    "n_retrieved_chunks":    int(len(retrieved)),
                    "retrieved_token_count": int(count_retrieved_tokens(retrieved)),
                    "prompt_token_count":    int(count_text_tokens(result["prompt"])),
                    "answer_token_count":    int(count_text_tokens(result["answer"])),
                    "energy_wh_per_query":   None,
                    "co2_g_per_query":       None,
                    "retrieved_chunk_ids":   ";".join(rr.chunk.chunk_id for rr in retrieved),
                    "answer_text":           result["answer"],
                    "notes":                 "",
                })
        finally:
            qry_co2_kg = qry_tracker.stop()
        rep_wall_s = time.perf_counter() - rep_t0

        qry_energy_wh = qry_tracker.final_emissions_data.energy_consumed * 1000.0
        qry_co2_g     = qry_co2_kg * 1000.0

        allocate_block_energy_proportionally(
            rows, qry_energy_wh, qry_co2_g,
        )

        qry_csv = RESULTS_DIR / "query_log.csv"
        write_header = not qry_csv.exists()
        pd.DataFrame(rows, columns=EXPECTED_QUERY_COLS).to_csv(
            qry_csv, mode="a", index=False, header=write_header,
        )

        n_refusals = sum(
            1 for row in rows if REFUSAL_PHRASE in row["answer_text"]
        )
        mean_total_ms = sum(row["total_latency_ms"] for row in rows) / len(rows)
        rep_summaries.append({
            "rep": r,
            "qry_run_id": qry_run_id,
            "energy_wh": qry_energy_wh,
            "co2_g": qry_co2_g,
            "n_refusals": n_refusals,
            "mean_total_ms": mean_total_ms,
            "rep_wall_s": rep_wall_s,
        })
        print(f"[D.2]   rep {r}: wall={rep_wall_s:.1f}s "
              f"mean_total_latency={mean_total_ms/1000:.1f}s "
              f"refusals={n_refusals}/13 "
              f"qry_energy={qry_energy_wh:.4f} Wh  "
              f"qry_run_id={qry_run_id}")

    # 3) Sanity asserts after all 3 reps for this config.
    idx_df_full = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
    qry_df_full = pd.read_csv(RESULTS_DIR / "query_log.csv")
    assert_indexing_schema(idx_df_full)
    assert_query_schema(qry_df_full)

    idx_this = idx_df_full[idx_df_full["run_id"] == idx_run_id]
    qry_this = qry_df_full[
        qry_df_full["run_id"].astype(str).str.startswith(idx_run_id)
    ]
    assert_query_sanity(
        qry_this, idx_this,
        top_k=cfg.top_k,
        chunk_size=cfg.chunk_size,
        expected_query_count=39,
        expected_indexing_count=1,
        notes_tag=NOTES_TAG_PROP_ENERGY,
    )
    print(f"[D.2] Sanity asserts (schema lock + A–H) PASSED for "
          f"hash={cfg.hash} (39 query rows, 1 indexing row)")

    return {
        "idx_run_id":         idx_run_id,
        "config_hash":        cfg.hash,
        "embedding_model":    cfg.embedding_model,
        "reused":             reuse_artifact,
        "n_chunks_total":     n_chunks_for_summary,
        "indexing_time_sec":  INDEXING_TIME_SEC,
        "peak_ram_mb":        peak_ram_mb_for_idx,
        "index_size_mb":      INDEX_SIZE_MB,
        "idx_energy_wh":      idx_energy_wh,
        "idx_co2_g":          idx_co2_g,
        "rep_summaries":      rep_summaries,
    }


In [5]:
"""Cell 6 — Sweep the 3 Exp2 configs back-to-back."""

all_exp2_results: list[dict] = []
total_t0 = time.perf_counter()

for cfg, reuse in zip(configs_to_run, reuse_decisions):
    summary = run_one_exp2_config(cfg, reuse_artifact=reuse)
    all_exp2_results.append(summary)

total_wall_s = time.perf_counter() - total_t0
print()
print("#" * 78)
print(f"# Exp2 Run All complete: total wall = {total_wall_s/60:.1f} min "
      f"({total_wall_s:.0f} s) for {len(all_exp2_results)} configs")
print("#" * 78)



[D.2] Config: embedding=sentence-transformers/all-mpnet-base-v2
      chunk_size=512, overlap_pct=20%, hash=d7129fb0, reuse_artifact=False
      idx_run_id = 20260510T110155Z_d7129fb0
[D.2] RAM pre-flight: used_now=9403 MB, model_est=460, spike_buffer=500 -> proj_used=10363 MB (threshold 12288 MB), proj_available=6021 MB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[D.2] Indexed: n_chunks=821, indexing_time=115.46s, peak_ram=1232.2 MB, index_size=2.52 MB, energy=1.4590 Wh, tracemalloc_peak=29.38 MB (Q4: not in CSV)
[D.2]   rep 1: wall=189.8s mean_total_latency=14.6s refusals=4/13 qry_energy=2.3994 Wh  qry_run_id=20260510T110155Z_d7129fb0_r1
[D.2]   rep 2: wall=216.4s mean_total_latency=16.6s refusals=4/13 qry_energy=2.7344 Wh  qry_run_id=20260510T110155Z_d7129fb0_r2
[D.2]   rep 3: wall=216.2s mean_total_latency=16.6s refusals=4/13 qry_energy=2.7328 Wh  qry_run_id=20260510T110155Z_d7129fb0_r3
[D.2] Sanity asserts (schema lock + A–H) PASSED for hash=d7129fb0 (39 query rows, 1 indexing row)

[D.2] Config: embedding=BAAI/bge-large-en-v1.5
      chunk_size=512, overlap_pct=20%, hash=4d57f78b, reuse_artifact=False
      idx_run_id = 20260510T111414Z_4d57f78b
[D.2] RAM pre-flight: used_now=13424 MB, model_est=1400, spike_buffer=500 -> proj_used=15324 MB (threshold 12288 MB), proj_available=1060 MB


RuntimeError: RAM pre-flight HALT for BAAI/bge-large-en-v1.5: used_now=13424 MB, model_est=1400, spike_buffer=500 -> proj_used=15324 MB (threshold 12288 MB), proj_available=1060 MB

In [6]:
"""Cell 7 — Combined Exp2 summary + cross-run winner computation.

Two sections:

  Section 1 — per-config detail for THIS Run All only. Reads from the
  in-memory all_exp2_results list. Useful when the run is unattended and
  the user wants a quick at-the-bottom-of-the-output summary of what
  just happened (without needing to re-load CSVs).

  Section 2 — cross-run Exp2 Pareto ranking. Reads ALL phase_d_exp2 rows
  from results/{indexing,query}_log.csv on disk. This is robust to
  partial Run-Alls: if a previous Run All persisted some Exp2 configs
  before being interrupted (e.g. by a RAM pre-flight halt), those rows
  are already on disk and are included in the ranking automatically.
  This is the AUTHORITATIVE Exp2 winner — Section 1's print only covers
  what was just executed.

Winner rule (mirrors Exp1, locked):
  primary       refusal_rate          ASC
  tie-break #1  mean_total_latency_ms ASC
  tie-break #2  indexing_time_sec     ASC
"""

# ----- Section 1 -----------------------------------------------------------
print()
print("=" * 78)
print(f"Phase D / Exp2 — per-config summary for THIS Run All  "
      f"({len(all_exp2_results)} configs)")
print("=" * 78)

for s in all_exp2_results:
    n_ref_per_rep    = [rep["n_refusals"]    for rep in s["rep_summaries"]]
    mean_lat_per_rep = [rep["mean_total_ms"] for rep in s["rep_summaries"]]
    energy_per_rep   = [rep["energy_wh"]     for rep in s["rep_summaries"]]
    mean_refusal_rate     = sum(n_ref_per_rep) / (3 * 13)
    mean_total_latency_ms = sum(mean_lat_per_rep) / 3.0
    print()
    print(f"  config: hash={s['config_hash']}  embedding={s['embedding_model']}")
    print(f"    reused={s['reused']}  n_chunks_total={s['n_chunks_total']}  "
          f"indexing_time={s['indexing_time_sec']:.2f}s  "
          f"peak_ram_mb={s['peak_ram_mb']:.1f}  "
          f"index_size_mb={s['index_size_mb']:.2f}")
    print(f"    idx_energy_wh={s['idx_energy_wh']:.4f}  "
          f"idx_co2_g={s['idx_co2_g']:.4f}")
    print(f"    refusals/rep   : {n_ref_per_rep}  "
          f"-> mean_refusal_rate={mean_refusal_rate:.1%}")
    print(f"    mean_total_ms/rep: {[f'{x:.0f}' for x in mean_lat_per_rep]}  "
          f"-> mean={mean_total_latency_ms:.0f}")
    print(f"    qry_energy_wh/rep: {[f'{x:.4f}' for x in energy_per_rep]}")

# ----- Section 2 -----------------------------------------------------------
print()
print("=" * 78)
print("Phase D / Exp2 — CROSS-RUN Pareto ranking (loads all phase_d_exp2 rows)")
print("=" * 78)

idx_full = pd.read_csv(RESULTS_DIR / "indexing_log.csv")
qry_full = pd.read_csv(RESULTS_DIR / "query_log.csv")

# Match rows whose notes column STARTS with "phase_d_exp2" — this catches
# both the plain tag and the historical reuse-tag variant
# "phase_d_exp2 (reused phase_d_exp1 ...)".
exp2_idx = idx_full[
    idx_full["notes"].astype(str).str.startswith("phase_d_exp2")
].copy()
print(f"Found {len(exp2_idx)} phase_d_exp2 indexing rows in CSV")
for _, irow in exp2_idx.iterrows():
    print(f"  hash={irow['config_hash']}  run_id={irow['run_id']}  "
          f"embedding={irow['embedding_model']}")

ranked_rows: list[dict] = []
for _, irow in exp2_idx.iterrows():
    idx_run_id = irow["run_id"]
    qsel = qry_full[
        qry_full["run_id"].astype(str).str.startswith(idx_run_id)
    ]
    if len(qsel) != 39:
        print(f"  WARNING: hash={irow['config_hash']} has {len(qsel)} query rows, "
              f"expected 39 — excluding from ranking.")
        continue
    n_ref = qsel["answer_text"].astype(str).str.contains(
        REFUSAL_PHRASE, regex=False,
    ).sum()
    refusal_rate = n_ref / 39.0
    mean_total_latency_ms = float(qsel["total_latency_ms"].mean())
    ranked_rows.append({
        "config_hash":           irow["config_hash"],
        "embedding_model":       irow["embedding_model"],
        "refusal_rate":          float(refusal_rate),
        "n_refusals":            int(n_ref),
        "mean_total_latency_ms": mean_total_latency_ms,
        "indexing_time_sec":     float(irow["indexing_time_sec"]),
    })

ranked = pd.DataFrame(ranked_rows).sort_values(
    ["refusal_rate", "mean_total_latency_ms", "indexing_time_sec"],
    kind="mergesort",
).reset_index(drop=True)

print()
header = (f"{'rank':>4}  {'hash':<10}  {'embedding':<48}  "
          f"{'refusal':>8}  {'n_ref/39':>8}  "
          f"{'mean_lat_ms':>11}  {'idx_time_s':>10}")
print(header)
print("-" * len(header))
for r, row in ranked.iterrows():
    print(f"{r+1:>4}  {row.config_hash:<10}  {row.embedding_model:<48}  "
          f"{row.refusal_rate:>7.1%}  {row.n_refusals:>8}  "
          f"{row.mean_total_latency_ms:>11.0f}  {row.indexing_time_sec:>10.2f}")

assert len(ranked) >= 1, "FAIL: no Exp2 configs found for ranking"
winner = ranked.iloc[0]
print()
print("=" * 78)
print("EXP2 WINNER  (cross-run, authoritative)")
print("=" * 78)
print(f"  config_hash:           {winner.config_hash}")
print(f"  embedding_model:       {winner.embedding_model}")
print(f"  mean refusal rate:     {winner.refusal_rate:.1%}  ({winner.n_refusals}/39)")
print(f"  mean total_latency_ms: {winner.mean_total_latency_ms:.0f}")
print(f"  indexing_time_sec:     {winner.indexing_time_sec:.2f}")

if len(ranked) < 3:
    print()
    print(f"  NOTE: only {len(ranked)} configs ranked — Exp2 expects 3 "
          f"(MiniLM, MPNet, BGE). If this is a partial Run All, finish the "
          f"sweep before treating this winner as final.")



Phase D / Exp2 — per-config summary for THIS Run All  (1 configs)

  config: hash=d7129fb0  embedding=sentence-transformers/all-mpnet-base-v2
    reused=False  n_chunks_total=821  indexing_time=115.46s  peak_ram_mb=1232.2  index_size_mb=2.52
    idx_energy_wh=1.4590  idx_co2_g=0.5558
    refusals/rep   : [4, 4, 4]  -> mean_refusal_rate=30.8%
    mean_total_ms/rep: ['14602', '16641', '16632']  -> mean=15958
    qry_energy_wh/rep: ['2.3994', '2.7344', '2.7328']

Phase D / Exp2 — CROSS-RUN Pareto ranking (loads all phase_d_exp2 rows)
Found 2 phase_d_exp2 indexing rows in CSV
  hash=aadb0cb9  run_id=20260510T091735Z_aadb0cb9  embedding=sentence-transformers/all-MiniLM-L6-v2
  hash=d7129fb0  run_id=20260510T110155Z_d7129fb0  embedding=sentence-transformers/all-mpnet-base-v2

rank  hash        embedding                                          refusal  n_ref/39  mean_lat_ms  idx_time_s
----------------------------------------------------------------------------------------------------------

## Wrap-up — what this notebook produced

For each of the 3 embedding models swept against the frozen Exp1 winner
(`chunk_size=512`, `overlap_pct=20%`, `top_k=3`):

- **1 row** appended to `results/indexing_log.csv` (`notes="phase_d_exp2"`
  for fresh builds; `notes="phase_d_exp2 (reused phase_d_exp1 ...)"` for
  the MiniLM reuse case).
- **39 rows** appended to `results/query_log.csv` (3 reps × 13 questions,
  each tagged with `NOTES_TAG_PROP_ENERGY` per the Phase C Q2 convention).
- **One indexing emission block + three query emission blocks** in
  `results/carbon/emissions.csv` per fresh-build config
  (`project_name=phase_d_exp2_indexing_<hash>` and
   `phase_d_exp2_query_<hash>_r{r}`). The reused MiniLM config has 0
  indexing emission blocks (no tracker started) + 3 query blocks.
- **Per-config sanity asserts** — schema lock-in on both CSVs, plus A–H
  value asserts (`src.metrics.assert_query_sanity`) on rows filtered by
  `run_id.str.startswith(idx_run_id)`.

The Exp2 winner's `embedding_model` is then frozen as the Exp3 baseline
in `notebooks/05_exp3_topk_depth.ipynb`.
